# Round 2: reliability of the Phase 3 results, and few-shot PatchCore

Run on a Colab T4 GPU, top to bottom. First run with `QUICK = True` (a short check of the whole
pipeline), then set `QUICK = False` and run everything again for the real results.

Every setting is fixed before the run and matches Phase 3: 256x256 images, WideResNet50-2,
PatchCore layers 2+3 with coreset 0.1, PaDiM n_features=300, threshold = 99th percentile of
validation image scores. No setting is chosen from test results, and every run is reported.

The MVTec test set was already used in Phase 3, so the few-shot results on MVTec are exploratory.

## 1. Save project code

In [ ]:
%%writefile prepare_dataset.py
"""Validate the MVTec AD Pill folder and write a reproducible train/validation split.

Uses only the standard library so it runs anywhere, including Colab.

    python src/prepare_dataset.py --root /path/to/mvtec --out splits/pill_seed42.json
"""
import argparse
import json
import random
import sys
from pathlib import Path

EXPECTED_TRAIN_GOOD = 267
EXPECTED_TEST_GOOD = 26
EXPECTED_TEST_DEFECT = 141
EXPECTED_DEFECTS = {
    "color",
    "combined",
    "contamination",
    "crack",
    "faulty_imprint",
    "pill_type",
    "scratch",
}


class DatasetError(Exception):
    """The dataset folder does not match the expected structure."""


def find_category_root(root, category="pill"):
    """Return the category folder, whether root is that folder or a parent of it."""
    root = Path(root)
    if (root / "train").is_dir() and (root / "test").is_dir():
        return root
    for candidate in sorted(root.rglob(category)):
        if (candidate / "train").is_dir() and (candidate / "test").is_dir():
            return candidate
    raise DatasetError(f"No '{category}' folder with train/ and test/ under {root}")


def _pngs(folder):
    return sorted(p.name for p in folder.glob("*.png"))


def inspect_dataset(pill_root):
    """Count images and check every defect image has a mask. Raises DatasetError."""
    pill_root = Path(pill_root)
    problems = []

    train_good = _pngs(pill_root / "train" / "good")
    test_good = _pngs(pill_root / "test" / "good")
    test_dirs = {p.name for p in (pill_root / "test").iterdir() if p.is_dir()}
    defects = sorted(test_dirs - {"good"})

    if set(defects) != EXPECTED_DEFECTS:
        problems.append(
            f"defect folders differ: missing {sorted(EXPECTED_DEFECTS - set(defects))}, "
            f"unexpected {sorted(set(defects) - EXPECTED_DEFECTS)}"
        )

    defect_counts = {}
    for defect in defects:
        images = _pngs(pill_root / "test" / defect)
        defect_counts[defect] = len(images)
        mask_dir = pill_root / "ground_truth" / defect
        masks = set(_pngs(mask_dir)) if mask_dir.is_dir() else set()
        expected_masks = {f"{Path(name).stem}_mask.png" for name in images}
        missing = sorted(expected_masks - masks)
        if missing:
            problems.append(f"{defect}: {len(missing)} missing masks, e.g. {missing[0]}")

    total_defect = sum(defect_counts.values())
    for label, actual, expected in (
        ("train/good", len(train_good), EXPECTED_TRAIN_GOOD),
        ("test/good", len(test_good), EXPECTED_TEST_GOOD),
        ("test defects", total_defect, EXPECTED_TEST_DEFECT),
    ):
        if actual != expected:
            problems.append(f"{label}: expected {expected} images, found {actual}")

    if problems:
        raise DatasetError("; ".join(problems))

    return {
        "train_good": len(train_good),
        "test_good": len(test_good),
        "test_defect": total_defect,
        "defect_counts": defect_counts,
    }


def make_split(pill_root, val_ratio=0.15, seed=42):
    """Split train/good only. The test set is never touched."""
    if not 0 < val_ratio < 1:
        raise ValueError("val_ratio must be between 0 and 1")
    names = _pngs(Path(pill_root) / "train" / "good")
    random.Random(seed).shuffle(names)
    n_val = round(len(names) * val_ratio)
    return {
        "seed": seed,
        "val_ratio": val_ratio,
        "train": sorted(names[n_val:]),
        "validation": sorted(names[:n_val]),
    }


def make_pillqc_split(groups, seed=42):
    """Split pillQC normal crops by source photo, so one photo never feeds two sets.

    groups: {photo name: [crop file names]}. Two photos go to test, two to
    validation, and the rest form the pool that memory banks are built from.
    """
    photos = sorted(groups)
    if len(photos) < 5:
        raise ValueError("need at least 5 source photos: 2 test, 2 validation, 1 pool")
    random.Random(seed).shuffle(photos)
    parts = {"test": photos[:2], "validation": photos[2:4], "pool": photos[4:]}
    split = {"seed": seed, "method": "by_source_photo", "photos": parts}
    for part, names in parts.items():
        split[part] = sorted(crop for name in names for crop in groups[name])
    return split


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--root", required=True, help="pill folder or a parent of it")
    parser.add_argument("--out", default="splits/pill_seed42.json")
    parser.add_argument("--val-ratio", type=float, default=0.15)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args(argv)

    try:
        pill_root = find_category_root(args.root)
        summary = inspect_dataset(pill_root)
    except DatasetError as error:
        print(f"Dataset check failed: {error}", file=sys.stderr)
        return 1

    split = make_split(pill_root, args.val_ratio, args.seed)
    out = Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(split, indent=2) + "\n")

    print(json.dumps(summary, indent=2))
    print(f"train={len(split['train'])} validation={len(split['validation'])} -> {out}")
    return 0


if __name__ == "__main__":
    sys.exit(main())

In [ ]:
%%writefile ad_metrics.py
"""Evaluation metrics for anomaly detection. Model independent."""
import numpy as np
from scipy import ndimage
from sklearn.metrics import roc_auc_score


def pick_threshold(validation_scores, quantile=0.99):
    """Image-score threshold taken from normal validation images only."""
    return float(np.quantile(np.asarray(validation_scores, dtype=np.float64), quantile))


def au_pro(masks, maps, fpr_limit=0.3, steps=200):
    """Area under the per-region-overlap curve up to fpr_limit, scaled to 0-1.

    masks: bool array (N, H, W). maps: float array (N, H, W).
    Each connected defect region counts equally, however large it is.
    """
    masks = np.asarray(masks, dtype=bool)
    maps = np.asarray(maps, dtype=np.float32)
    if masks.shape != maps.shape:
        raise ValueError("masks and maps must have the same shape")
    if not masks.any():
        raise ValueError("au_pro needs at least one defect pixel")

    fprs = np.linspace(0.0, fpr_limit, steps + 1)
    thresholds = np.quantile(maps[~masks], 1.0 - fprs)

    overlaps = []
    for mask, score_map in zip(masks, maps):
        labels, count = ndimage.label(mask)
        for region in range(1, count + 1):
            scores = np.sort(score_map[labels == region])
            below = np.searchsorted(scores, thresholds, side="right")
            overlaps.append(1.0 - below / scores.size)
    pro = np.mean(overlaps, axis=0)

    area = np.sum((pro[1:] + pro[:-1]) / 2 * np.diff(fprs))
    return float(area / fpr_limit)


def image_metrics(labels, scores, threshold):
    """Image-level metrics only, for datasets without masks. labels: 1 means defective."""
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=np.float64)
    predicted = scores > threshold

    true_positive = int(np.sum(predicted & (labels == 1)))
    false_positive = int(np.sum(predicted & (labels == 0)))
    false_negative = int(np.sum(~predicted & (labels == 1)))
    true_negative = int(np.sum(~predicted & (labels == 0)))

    precision = true_positive / max(true_positive + false_positive, 1)
    recall = true_positive / max(true_positive + false_negative, 1)
    return {
        "image_auroc": float(roc_auc_score(labels, scores)),
        "threshold": float(threshold),
        "precision": precision,
        "recall": recall,
        "f1": 2 * precision * recall / max(precision + recall, 1e-12),
        "true_positive": true_positive,
        "false_positive": false_positive,
        "false_negative": false_negative,
        "true_negative": true_negative,
    }


def evaluate(labels, scores, masks, maps, threshold):
    """Image and pixel metrics for one model. labels: 1 means defective."""
    masks = np.asarray(masks, dtype=bool)
    image = image_metrics(labels, scores, threshold)
    return {
        "image_auroc": image.pop("image_auroc"),
        "pixel_auroc": float(roc_auc_score(masks.ravel(), np.asarray(maps).ravel())),
        "au_pro": au_pro(masks, maps),
        **image,
    }


def evaluate_by_defect(groups, scores, masks, maps, threshold):
    """Per-defect results. Each defect is compared against the good images."""
    groups = np.asarray(groups)
    scores = np.asarray(scores, dtype=np.float64)
    masks = np.asarray(masks, dtype=bool)
    maps = np.asarray(maps)
    good = groups == "good"

    rows = {}
    for defect in sorted(set(groups.tolist()) - {"good"}):
        selected = groups == defect
        both = selected | good
        rows[defect] = {
            "images": int(selected.sum()),
            "image_auroc": float(roc_auc_score(selected[both], scores[both])),
            "pixel_auroc": float(roc_auc_score(masks[both].ravel(), maps[both].ravel())),
            "au_pro": au_pro(masks[both], maps[both]),
            "recall": float(np.mean(scores[selected] > threshold)),
        }
    return rows

In [ ]:
%%writefile bootstrap.py
"""Bootstrap confidence intervals for image-level anomaly detection metrics.

Images are resampled with replacement, normal and defective separately, so every
resample keeps both classes. The threshold is held fixed, so the intervals do not
include the uncertainty of the threshold itself.
"""
import numpy as np
from sklearn.metrics import roc_auc_score

import ad_metrics

METRICS = ("image_auroc", "precision", "recall", "f1")


def _resamples(labels, rounds, seed):
    normal, defect = np.flatnonzero(labels == 0), np.flatnonzero(labels == 1)
    if normal.size == 0 or defect.size == 0:
        raise ValueError("bootstrap needs both normal and defective images")
    rng = np.random.default_rng(seed)
    for _ in range(rounds):
        yield np.concatenate([rng.choice(normal, normal.size), rng.choice(defect, defect.size)])


def _interval(value, draws, level):
    tail = (1 - level) / 2
    low, high = np.quantile(draws, [tail, 1 - tail])
    return {"value": float(value), "low": float(low), "high": float(high)}


def image_ci(labels, scores, threshold, rounds=2000, seed=42, level=0.95):
    """Percentile interval of each image metric. labels: 1 means defective."""
    labels = np.asarray(labels, dtype=int)
    scores = np.asarray(scores, dtype=np.float64)
    point = ad_metrics.image_metrics(labels, scores, threshold)
    draws = [
        ad_metrics.image_metrics(labels[index], scores[index], threshold)
        for index in _resamples(labels, rounds, seed)
    ]
    return {name: _interval(point[name], [draw[name] for draw in draws], level) for name in METRICS}


def paired_auroc_diff_ci(labels, scores_a, scores_b, rounds=2000, seed=42, level=0.95):
    """Interval of image AUROC(a) minus AUROC(b), both scored on the same resamples."""
    labels = np.asarray(labels, dtype=int)
    scores_a = np.asarray(scores_a, dtype=np.float64)
    scores_b = np.asarray(scores_b, dtype=np.float64)
    point = roc_auc_score(labels, scores_a) - roc_auc_score(labels, scores_b)
    draws = [
        roc_auc_score(labels[index], scores_a[index]) - roc_auc_score(labels[index], scores_b[index])
        for index in _resamples(labels, rounds, seed)
    ]
    return _interval(point, draws, level)

## 2. Settings fixed before the run

In [ ]:
QUICK = True  # True: a short check of the whole pipeline. Set False for the real run.

SEEDS = [42, 0, 1, 2, 3]  # Seeds the model, and the draw of few-shot images.
SHOTS = [1, 5, 10, 25]  # Normal images per memory bank. "All" is added for each dataset.
BOOTSTRAP_ROUNDS = 2000
SIZE = 256
BACKBONE = "wide_resnet50_2"
PARITY_TOLERANCE = 0.005  # Seed 42 must reproduce the Phase 3 image AUROC this closely.
PHASE3_IMAGE_AUROC = {"patchcore": 0.9495362793235135, "padim": 0.916530278232406}

if QUICK:
    SEEDS, SHOTS, BOOTSTRAP_ROUNDS = [42], [1, 5], 100

## 3. Install and load MVTec Pill

In [ ]:
%pip install -q anomalib==2.7.0 kagglehub

In [ ]:
import gc, json, os, random, shutil, warnings
from pathlib import Path

import kagglehub
import numpy as np
import torch
from PIL import Image
from sklearn.metrics import roc_auc_score
from torchvision.transforms.v2 import Compose, Normalize, Resize
from torchvision.transforms.v2.functional import pil_to_tensor, to_dtype

import ad_metrics
import bootstrap
import prepare_dataset

warnings.filterwarnings("ignore")
SPLIT = json.loads('''{"seed": 42, "val_ratio": 0.15, "train": ["000.png", "001.png", "002.png", "004.png", "006.png", "007.png", "008.png", "010.png", "011.png", "012.png", "013.png", "014.png", "015.png", "016.png", "017.png", "018.png", "019.png", "020.png", "021.png", "022.png", "023.png", "024.png", "025.png", "026.png", "027.png", "028.png", "030.png", "031.png", "032.png", "033.png", "034.png", "035.png", "036.png", "037.png", "038.png", "039.png", "040.png", "041.png", "042.png", "043.png", "044.png", "045.png", "046.png", "047.png", "048.png", "049.png", "050.png", "052.png", "053.png", "054.png", "055.png", "056.png", "057.png", "058.png", "059.png", "060.png", "061.png", "062.png", "063.png", "064.png", "067.png", "068.png", "069.png", "070.png", "071.png", "073.png", "074.png", "075.png", "076.png", "077.png", "079.png", "080.png", "081.png", "082.png", "083.png", "084.png", "086.png", "087.png", "088.png", "089.png", "090.png", "091.png", "092.png", "093.png", "094.png", "095.png", "096.png", "097.png", "098.png", "099.png", "100.png", "101.png", "102.png", "104.png", "105.png", "107.png", "108.png", "109.png", "110.png", "111.png", "113.png", "114.png", "115.png", "116.png", "117.png", "118.png", "120.png", "121.png", "124.png", "125.png", "126.png", "127.png", "128.png", "129.png", "130.png", "132.png", "133.png", "134.png", "135.png", "136.png", "137.png", "139.png", "140.png", "141.png", "142.png", "143.png", "145.png", "146.png", "147.png", "148.png", "150.png", "151.png", "152.png", "154.png", "155.png", "156.png", "157.png", "158.png", "159.png", "160.png", "161.png", "162.png", "163.png", "164.png", "165.png", "166.png", "167.png", "168.png", "169.png", "170.png", "171.png", "172.png", "173.png", "174.png", "175.png", "176.png", "178.png", "179.png", "180.png", "181.png", "182.png", "183.png", "185.png", "186.png", "187.png", "188.png", "189.png", "190.png", "191.png", "192.png", "193.png", "194.png", "195.png", "196.png", "197.png", "200.png", "202.png", "203.png", "204.png", "205.png", "206.png", "207.png", "208.png", "209.png", "211.png", "212.png", "213.png", "214.png", "216.png", "217.png", "218.png", "219.png", "220.png", "222.png", "223.png", "226.png", "227.png", "228.png", "229.png", "230.png", "231.png", "232.png", "233.png", "234.png", "235.png", "236.png", "237.png", "238.png", "239.png", "240.png", "242.png", "243.png", "245.png", "249.png", "251.png", "252.png", "253.png", "254.png", "255.png", "256.png", "257.png", "258.png", "259.png", "261.png", "262.png", "264.png", "265.png"], "validation": ["003.png", "005.png", "009.png", "029.png", "051.png", "065.png", "066.png", "072.png", "078.png", "085.png", "103.png", "106.png", "112.png", "119.png", "122.png", "123.png", "131.png", "138.png", "144.png", "149.png", "153.png", "177.png", "184.png", "198.png", "199.png", "201.png", "210.png", "215.png", "221.png", "224.png", "225.png", "241.png", "244.png", "246.png", "247.png", "248.png", "250.png", "260.png", "263.png", "266.png"]}''')
PILLQC_SPLIT = json.loads('''{"seed": 42, "method": "by_source_photo", "photos": {"test": ["normal5", "normal3"], "validation": ["normal4", "normal6"], "pool": ["normal2", "normal7"]}, "test": ["normal0020.jpg", "normal0021.jpg", "normal0022.jpg", "normal0023.jpg", "normal0024.jpg", "normal0025.jpg", "normal0026.jpg", "normal0027.jpg", "normal0028.jpg", "normal0029.jpg", "normal0030.jpg", "normal0031.jpg", "normal0032.jpg", "normal0033.jpg", "normal0034.jpg", "normal0035.jpg", "normal0036.jpg", "normal0037.jpg", "normal0038.jpg", "normal0039.jpg", "normal0040.jpg", "normal0041.jpg", "normal0069.jpg", "normal0070.jpg", "normal0071.jpg", "normal0072.jpg", "normal0073.jpg", "normal0074.jpg", "normal0075.jpg", "normal0076.jpg", "normal0077.jpg", "normal0078.jpg", "normal0079.jpg", "normal0080.jpg", "normal0081.jpg", "normal0082.jpg", "normal0083.jpg", "normal0084.jpg", "normal0085.jpg", "normal0086.jpg", "normal0087.jpg", "normal0088.jpg", "normal0089.jpg", "normal0090.jpg", "normal0091.jpg", "normal0092.jpg", "normal0093.jpg", "normal0094.jpg", "normal0095.jpg"], "validation": ["normal0042.jpg", "normal0043.jpg", "normal0044.jpg", "normal0045.jpg", "normal0046.jpg", "normal0047.jpg", "normal0048.jpg", "normal0049.jpg", "normal0050.jpg", "normal0051.jpg", "normal0052.jpg", "normal0053.jpg", "normal0054.jpg", "normal0055.jpg", "normal0056.jpg", "normal0057.jpg", "normal0058.jpg", "normal0059.jpg", "normal0060.jpg", "normal0061.jpg", "normal0062.jpg", "normal0063.jpg", "normal0064.jpg", "normal0065.jpg", "normal0066.jpg", "normal0067.jpg", "normal0068.jpg", "normal0096.jpg", "normal0097.jpg", "normal0098.jpg", "normal0099.jpg", "normal0100.jpg", "normal0101.jpg", "normal0102.jpg", "normal0103.jpg", "normal0104.jpg", "normal0105.jpg", "normal0106.jpg", "normal0107.jpg", "normal0108.jpg", "normal0109.jpg", "normal0110.jpg", "normal0111.jpg", "normal0112.jpg", "normal0113.jpg", "normal0114.jpg", "normal0115.jpg", "normal0116.jpg", "normal0117.jpg", "normal0118.jpg", "normal0119.jpg", "normal0120.jpg", "normal0121.jpg", "normal0122.jpg"], "pool": ["normal0001.jpg", "normal0002.jpg", "normal0003.jpg", "normal0004.jpg", "normal0005.jpg", "normal0006.jpg", "normal0007.jpg", "normal0008.jpg", "normal0009.jpg", "normal0010.jpg", "normal0011.jpg", "normal0012.jpg", "normal0013.jpg", "normal0014.jpg", "normal0015.jpg", "normal0016.jpg", "normal0017.jpg", "normal0018.jpg", "normal0019.jpg", "normal0123.jpg", "normal0124.jpg", "normal0125.jpg", "normal0126.jpg", "normal0127.jpg", "normal0128.jpg", "normal0129.jpg", "normal0130.jpg", "normal0131.jpg", "normal0132.jpg", "normal0133.jpg", "normal0134.jpg", "normal0135.jpg", "normal0136.jpg", "normal0137.jpg", "normal0138.jpg", "normal0139.jpg", "normal0140.jpg", "normal0141.jpg", "normal0142.jpg", "normal0143.jpg", "normal0144.jpg", "normal0145.jpg", "normal0146.jpg", "normal0147.jpg", "normal0148.jpg", "normal0149.jpg"], "sources": {"normal2": ["normal0001.jpg", "normal0002.jpg", "normal0003.jpg", "normal0004.jpg", "normal0005.jpg", "normal0006.jpg", "normal0007.jpg", "normal0008.jpg", "normal0009.jpg", "normal0010.jpg", "normal0011.jpg", "normal0012.jpg", "normal0013.jpg", "normal0014.jpg", "normal0015.jpg", "normal0016.jpg", "normal0017.jpg", "normal0018.jpg", "normal0019.jpg"], "normal3": ["normal0020.jpg", "normal0021.jpg", "normal0022.jpg", "normal0023.jpg", "normal0024.jpg", "normal0025.jpg", "normal0026.jpg", "normal0027.jpg", "normal0028.jpg", "normal0029.jpg", "normal0030.jpg", "normal0031.jpg", "normal0032.jpg", "normal0033.jpg", "normal0034.jpg", "normal0035.jpg", "normal0036.jpg", "normal0037.jpg", "normal0038.jpg", "normal0039.jpg", "normal0040.jpg", "normal0041.jpg"], "normal4": ["normal0042.jpg", "normal0043.jpg", "normal0044.jpg", "normal0045.jpg", "normal0046.jpg", "normal0047.jpg", "normal0048.jpg", "normal0049.jpg", "normal0050.jpg", "normal0051.jpg", "normal0052.jpg", "normal0053.jpg", "normal0054.jpg", "normal0055.jpg", "normal0056.jpg", "normal0057.jpg", "normal0058.jpg", "normal0059.jpg", "normal0060.jpg", "normal0061.jpg", "normal0062.jpg", "normal0063.jpg", "normal0064.jpg", "normal0065.jpg", "normal0066.jpg", "normal0067.jpg", "normal0068.jpg"], "normal5": ["normal0069.jpg", "normal0070.jpg", "normal0071.jpg", "normal0072.jpg", "normal0073.jpg", "normal0074.jpg", "normal0075.jpg", "normal0076.jpg", "normal0077.jpg", "normal0078.jpg", "normal0079.jpg", "normal0080.jpg", "normal0081.jpg", "normal0082.jpg", "normal0083.jpg", "normal0084.jpg", "normal0085.jpg", "normal0086.jpg", "normal0087.jpg", "normal0088.jpg", "normal0089.jpg", "normal0090.jpg", "normal0091.jpg", "normal0092.jpg", "normal0093.jpg", "normal0094.jpg", "normal0095.jpg"], "normal6": ["normal0096.jpg", "normal0097.jpg", "normal0098.jpg", "normal0099.jpg", "normal0100.jpg", "normal0101.jpg", "normal0102.jpg", "normal0103.jpg", "normal0104.jpg", "normal0105.jpg", "normal0106.jpg", "normal0107.jpg", "normal0108.jpg", "normal0109.jpg", "normal0110.jpg", "normal0111.jpg", "normal0112.jpg", "normal0113.jpg", "normal0114.jpg", "normal0115.jpg", "normal0116.jpg", "normal0117.jpg", "normal0118.jpg", "normal0119.jpg", "normal0120.jpg", "normal0121.jpg", "normal0122.jpg"], "normal7": ["normal0123.jpg", "normal0124.jpg", "normal0125.jpg", "normal0126.jpg", "normal0127.jpg", "normal0128.jpg", "normal0129.jpg", "normal0130.jpg", "normal0131.jpg", "normal0132.jpg", "normal0133.jpg", "normal0134.jpg", "normal0135.jpg", "normal0136.jpg", "normal0137.jpg", "normal0138.jpg", "normal0139.jpg", "normal0140.jpg", "normal0141.jpg", "normal0142.jpg", "normal0143.jpg", "normal0144.jpg", "normal0145.jpg", "normal0146.jpg", "normal0147.jpg", "normal0148.jpg", "normal0149.jpg"]}}''')

PILL = prepare_dataset.find_category_root(kagglehub.dataset_download("ipythonx/mvtec-ad"))
print(prepare_dataset.inspect_dataset(PILL))

TRAIN_DIR = PILL / "train" / "good"
VAL_PATHS = [TRAIN_DIR / f for f in SPLIT["validation"]]
TEST_PATHS = sorted((PILL / "test").glob("*/*.png"))
GROUPS = [p.parent.name for p in TEST_PATHS]
LABELS = [int(g != "good") for g in GROUPS]
assert not set(SPLIT["train"]) & set(SPLIT["validation"])

WORK = Path("/content/work")
OUT = Path("/content/results")
OUT.mkdir(exist_ok=True)
CONFIG = {
    "quick": QUICK, "seeds": SEEDS, "shots": SHOTS, "bootstrap_rounds": BOOTSTRAP_ROUNDS,
    "size": SIZE, "backbone": BACKBONE, "test_images": [f"{g}/{p.name}" for g, p in zip(GROUPS, TEST_PATHS)],
}

## 4. Train and predict (same code path as Phase 3)

In [ ]:
from anomalib.data import Folder
from anomalib.engine import Engine
from anomalib.models import Padim, Patchcore
from anomalib.pre_processing import PreProcessor
from lightning.pytorch import seed_everything

DEVICE = "cuda"

# Batched matrix inverse crashes CUDA on the T4, so PaDiM inverts on the CPU.
_inverse = torch.linalg.inv


def cpu_inverse(matrices):
    return _inverse(matrices.cpu()).to(matrices.device)


torch.linalg.inv = cpu_inverse


def make_transform(size):
    return Compose([
        Resize((size, size), antialias=True),
        Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])


def build(name, size):
    common = dict(
        pre_processor=PreProcessor(transform=make_transform(size)),
        post_processor=False, evaluator=False, visualizer=False,
    )
    if name == "patchcore":
        return Patchcore(backbone=BACKBONE, layers=("layer2", "layer3"), coreset_sampling_ratio=0.1, **common)
    return Padim(backbone=BACKBONE, layers=["layer1", "layer2", "layer3"], n_features=300, **common)


def link_images(folder, paths):
    shutil.rmtree(folder, ignore_errors=True)
    folder.mkdir(parents=True)
    for path in paths:
        os.symlink(path, folder / path.name)


def train(name, train_paths, val_paths, seed):
    gc.collect()
    torch.cuda.empty_cache()
    shutil.rmtree("/content/runs", ignore_errors=True)  # Checkpoints of earlier runs would fill the disk.
    seed_everything(seed)
    link_images(WORK / "train_good", train_paths)
    link_images(WORK / "val_good", val_paths)
    datamodule = Folder(
        name="pill", root=WORK, normal_dir="train_good", normal_test_dir="val_good",
        train_batch_size=8, eval_batch_size=8, num_workers=2,
        test_split_mode="from_dir", val_split_mode="same_as_test",
    )
    model = build(name, SIZE)
    engine = Engine(default_root_dir=f"/content/runs/{name}", logger=False, accelerator="gpu", devices=1,
                    enable_progress_bar=False, num_sanity_val_steps=0)
    engine.fit(model=model, datamodule=datamodule)
    return model.to(DEVICE).eval()


def load_image(path):
    return to_dtype(pil_to_tensor(Image.open(path).convert("RGB")), torch.float32, scale=True)


@torch.inference_mode()
def predict(model, paths, size=SIZE, batch_size=8):
    transform = make_transform(size)
    scores, maps = [], []
    for start in range(0, len(paths), batch_size):
        batch = torch.stack([transform(load_image(p)) for p in paths[start:start + batch_size]]).to(DEVICE)
        output = model.model(batch)
        scores.append(output.pred_score.flatten().float().cpu().numpy())
        maps.append(output.anomaly_map.squeeze(1).float().cpu().numpy())
    return np.concatenate(scores), np.concatenate(maps)


def load_mask(path, size=SIZE):
    if path.parent.name == "good":
        return np.zeros((size, size), dtype=bool)
    mask_path = PILL / "ground_truth" / path.parent.name / f"{path.stem}_mask.png"
    return np.asarray(Image.open(mask_path).convert("L").resize((size, size), Image.NEAREST)) > 0


MASKS = np.stack([load_mask(p) for p in TEST_PATHS])
print(len(TEST_PATHS), MASKS.shape, sum(LABELS))

In [ ]:
MVTEC_KEYS = ["image_auroc", "pixel_auroc", "au_pro", "precision", "recall", "f1"]
IMAGE_KEYS = ["image_auroc", "precision", "recall", "f1"]


def draw(pool, shots, seed):
    """The first `shots` names of a seeded shuffle, so a smaller draw sits inside a larger one."""
    names = sorted(pool)
    random.Random(seed).shuffle(names)
    return sorted(names[:shots])


def spread(rows, keys):
    """Mean, sample standard deviation, and range of each metric over runs."""
    table = {}
    for key in keys:
        values = np.array([row["overall"][key] for row in rows], dtype=float)
        table[key] = {
            "mean": float(values.mean()),
            "std": float(values.std(ddof=1)) if len(values) > 1 else 0.0,
            "min": float(values.min()),
            "max": float(values.max()),
        }
    return table


def show(label, row, keys):
    print("RESULT", label, json.dumps({key: round(row["overall"][key], 4) for key in keys}))


def run_mvtec(name, train_files, seed):
    model = train(name, [TRAIN_DIR / f for f in train_files], VAL_PATHS, seed)
    val_scores, _ = predict(model, VAL_PATHS)
    threshold = ad_metrics.pick_threshold(val_scores)
    scores, maps = predict(model, TEST_PATHS)
    return {
        "model": name,
        "seed": seed,
        "shots": len(train_files),
        "overall": ad_metrics.evaluate(LABELS, scores, MASKS, maps, threshold),
        "by_defect": ad_metrics.evaluate_by_defect(GROUPS, scores, MASKS, maps, threshold),
        "train_files": list(train_files),
        "validation_scores": val_scores.tolist(),
        "test_scores": scores.tolist(),
    }

## 5. Section A: several seeds and bootstrap intervals on MVTec

The split stays fixed. The seed only changes the coreset selection of PatchCore and the random
feature choice of PaDiM, so PatchCore may barely move between seeds. Seed 42 must reproduce Phase 3.

In [ ]:
RELIABILITY = {"config": CONFIG, "runs": []}
for name in ("patchcore", "padim"):
    for seed in SEEDS:
        row = run_mvtec(name, SPLIT["train"], seed)
        RELIABILITY["runs"].append(row)
        show(f"{name} seed {seed}", row, MVTEC_KEYS)
        if seed == 42:
            gap = abs(row["overall"]["image_auroc"] - PHASE3_IMAGE_AUROC[name])
            assert gap <= PARITY_TOLERANCE, f"{name} seed 42 is {gap:.4f} away from Phase 3. Stop and check."

In [ ]:
def seed42(name):
    return next(row for row in RELIABILITY["runs"] if row["model"] == name and row["seed"] == 42)


RELIABILITY["across_seeds"] = {
    name: spread([row for row in RELIABILITY["runs"] if row["model"] == name], MVTEC_KEYS)
    for name in ("patchcore", "padim")
}

labels, groups = np.array(LABELS), np.array(GROUPS)
intervals = {
    name: bootstrap.image_ci(labels, seed42(name)["test_scores"], seed42(name)["overall"]["threshold"],
                             rounds=BOOTSTRAP_ROUNDS)
    for name in ("patchcore", "padim")
}
intervals["patchcore_minus_padim_image_auroc"] = bootstrap.paired_auroc_diff_ci(
    labels, seed42("patchcore")["test_scores"], seed42("padim")["test_scores"], rounds=BOOTSTRAP_ROUNDS
)
# Per-defect intervals are wide (pill_type has 9 images): read them as a trend only.
scores, threshold = np.array(seed42("patchcore")["test_scores"]), seed42("patchcore")["overall"]["threshold"]
intervals["patchcore_by_defect"] = {}
for defect in sorted(set(GROUPS) - {"good"}):
    keep = (groups == defect) | (groups == "good")
    result = bootstrap.image_ci(labels[keep], scores[keep], threshold, rounds=BOOTSTRAP_ROUNDS)
    intervals["patchcore_by_defect"][defect] = {
        "images": int((groups == defect).sum()), "image_auroc": result["image_auroc"], "recall": result["recall"],
    }
RELIABILITY["bootstrap_seed42"] = intervals
(OUT / "reliability_results.json").write_text(json.dumps(RELIABILITY, indent=1))

for name in ("patchcore", "padim"):
    for key, row in RELIABILITY["across_seeds"][name].items():
        print("RESULT seeds", name, key, f"{row['mean']:.4f} +- {row['std']:.4f} (min {row['min']:.4f}, max {row['max']:.4f})")
    for key, row in intervals[name].items():
        print("RESULT ci95", name, key, f"{row['value']:.4f} [{row['low']:.4f}, {row['high']:.4f}]")
row = intervals["patchcore_minus_padim_image_auroc"]
print("RESULT ci95 patchcore minus padim image_auroc", f"{row['value']:.4f} [{row['low']:.4f}, {row['high']:.4f}]")
for defect, rows in intervals["patchcore_by_defect"].items():
    print("RESULT ci95 patchcore", defect, rows["images"], "images",
          {key: f"{rows[key]['value']:.3f} [{rows[key]['low']:.3f}, {rows[key]['high']:.3f}]" for key in ("image_auroc", "recall")})

## 6. Section B: few-shot PatchCore on MVTec

Memory banks are built from 1, 5, 10, or 25 normal images drawn from the 227 training images.
The 40 validation images that set the threshold stay the same, so a k-shot run really needs
k + 40 normal images. Image AUROC needs no threshold and is the main few-shot result.

In [ ]:
FEWSHOT = {"config": CONFIG, "mvtec": [], "pillqc": []}
for shots in SHOTS:
    for seed in SEEDS:
        row = run_mvtec("patchcore", draw(SPLIT["train"], shots, seed), seed)
        FEWSHOT["mvtec"].append(row)
        show(f"mvtec {shots}-shot seed {seed}", row, MVTEC_KEYS)
# The "all" rows are the PatchCore runs of section A.
FEWSHOT["mvtec"] += [row for row in RELIABILITY["runs"] if row["model"] == "patchcore"]
(OUT / "fewshot_results.json").write_text(json.dumps(FEWSHOT, indent=1))

## 7. Section C: few-shot PatchCore on pillQC

pillQC has no masks, so only image-level results exist. The normal crops are split by source
photo: 2 photos for the pool, 2 for validation, 2 for test. All chip and dirt crops are test images.

In [ ]:
!test -d /content/pillQC || git clone --depth 1 -q https://github.com/matlab-deep-learning/pillQC /content/pillQC

In [ ]:
QC = Path("/content/pillQC/images")
QC_POOL = PILLQC_SPLIT["pool"]
QC_VAL = [QC / "normal" / f for f in PILLQC_SPLIT["validation"]]
QC_TEST = [QC / "normal" / f for f in PILLQC_SPLIT["test"]]
QC_TEST += sorted((QC / "chip").glob("*.jpg")) + sorted((QC / "dirt").glob("*.jpg"))
QC_GROUPS = np.array([p.parent.name for p in QC_TEST])
QC_LABELS = (QC_GROUPS != "normal").astype(int)
assert (QC_GROUPS == "chip").sum() == 43 and (QC_GROUPS == "dirt").sum() == 138
assert not set(QC_POOL) & (set(PILLQC_SPLIT["validation"]) | set(PILLQC_SPLIT["test"]))
QC_SHOTS = [shots for shots in SHOTS if shots < len(QC_POOL)] + [len(QC_POOL)]
print("pool", len(QC_POOL), "validation", len(QC_VAL), "test", len(QC_TEST), "shots", QC_SHOTS)


def run_pillqc(train_files, seed):
    model = train("patchcore", [QC / "normal" / f for f in train_files], QC_VAL, seed)
    val_scores, _ = predict(model, QC_VAL)
    threshold = ad_metrics.pick_threshold(val_scores)
    scores, maps = predict(model, QC_TEST)
    by_defect = {}
    for defect in ("chip", "dirt"):
        keep = (QC_GROUPS == defect) | (QC_GROUPS == "normal")
        by_defect[defect] = {
            "images": int((QC_GROUPS == defect).sum()),
            "image_auroc": float(roc_auc_score(QC_LABELS[keep], scores[keep])),
            "recall": float(np.mean(scores[QC_GROUPS == defect] > threshold)),
        }
    row = {
        "model": "patchcore",
        "seed": seed,
        "shots": len(train_files),
        "overall": ad_metrics.image_metrics(QC_LABELS, scores, threshold),
        "by_defect": by_defect,
        "train_files": list(train_files),
        "validation_scores": val_scores.tolist(),
        "test_scores": scores.tolist(),
    }
    return row, maps


for shots in QC_SHOTS:
    for seed in SEEDS:
        row, maps = run_pillqc(draw(QC_POOL, shots, seed), seed)
        FEWSHOT["pillqc"].append(row)
        show(f"pillqc {shots}-shot seed {seed}", row, IMAGE_KEYS)
        if shots == len(QC_POOL) and seed == SEEDS[0]:
            QC_MAPS, QC_ROW = maps, row
FEWSHOT["config"]["pillqc_test_images"] = [f"{p.parent.name}/{p.name}" for p in QC_TEST]
(OUT / "fewshot_results.json").write_text(json.dumps(FEWSHOT, indent=1))

Normal and defective pillQC crops come from different photos, so the model could separate them by
lighting instead of by the defect. Check by eye that the hot areas below sit on the chip or the dirt.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 8, figsize=(17, 6.6))
low, high = np.quantile(QC_MAPS, [0.01, 0.999])
for line, group in enumerate(("normal", "chip", "dirt")):
    for column, index in enumerate(np.flatnonzero(QC_GROUPS == group)[:4]):
        image = Image.open(QC_TEST[index]).convert("RGB").resize((SIZE, SIZE))
        axes[line, 2 * column].imshow(image)
        axes[line, 2 * column].set_title(f"{group} score {QC_ROW['test_scores'][index]:.1f}", fontsize=9)
        axes[line, 2 * column + 1].imshow(image)
        axes[line, 2 * column + 1].imshow(QC_MAPS[index], cmap="jet", alpha=0.5, vmin=low, vmax=high)
for axis in axes.ravel():
    axis.axis("off")
fig.suptitle(f"pillQC, all {len(QC_POOL)} pool images, seed {SEEDS[0]}, threshold {QC_ROW['overall']['threshold']:.1f}")
fig.tight_layout()
fig.savefig(OUT / "pillqc_heatmaps.png", dpi=110)
plt.show()

## 8. Section D: summary

In [ ]:
SUMMARY = {}
fig, axis = plt.subplots(figsize=(6.4, 4))
for dataset, keys in (("mvtec", MVTEC_KEYS), ("pillqc", IMAGE_KEYS)):
    levels = sorted({row["shots"] for row in FEWSHOT[dataset]})
    SUMMARY[dataset] = {
        str(shots): spread([row for row in FEWSHOT[dataset] if row["shots"] == shots], keys) for shots in levels
    }
    for shots in levels:
        cells = [f"{key} {value['mean']:.3f} +- {value['std']:.3f}" for key, value in SUMMARY[dataset][str(shots)].items()]
        print("RESULT", dataset, f"{shots}-shot", " | ".join(cells))
    auroc = [SUMMARY[dataset][str(shots)]["image_auroc"] for shots in levels]
    axis.errorbar(range(len(levels)), [a["mean"] for a in auroc], yerr=[a["std"] for a in auroc],
                  marker="o", capsize=3, label=dataset)
    for position, shots in enumerate(levels):
        axis.annotate(str(shots), (position, auroc[position]["mean"]), textcoords="offset points", xytext=(4, -12), fontsize=8)
axis.set_xticks(range(len(levels)))
axis.set_xticklabels([str(shots) for shots in levels[:-1]] + ["all"])
axis.set_xlabel("normal images in the memory bank")
axis.set_ylabel("image AUROC (mean and std over seeds)")
axis.legend()
fig.tight_layout()
fig.savefig(OUT / "fewshot_image_auroc.png", dpi=130)
plt.show()

FEWSHOT["summary"] = SUMMARY
(OUT / "fewshot_results.json").write_text(json.dumps(FEWSHOT, indent=1))

# Every planned run is present, and no few-shot image is a validation or test image.
assert len(RELIABILITY["runs"]) == 2 * len(SEEDS)
assert len(FEWSHOT["mvtec"]) == (len(SHOTS) + 1) * len(SEEDS)
assert len(FEWSHOT["pillqc"]) == len(QC_SHOTS) * len(SEEDS)
assert all(set(row["train_files"]) <= set(SPLIT["train"]) for row in FEWSHOT["mvtec"])
assert all(set(row["train_files"]) <= set(QC_POOL) for row in FEWSHOT["pillqc"])
print("runs", len(RELIABILITY["runs"]), len(FEWSHOT["mvtec"]), len(FEWSHOT["pillqc"]))

## 9. Save results

In [ ]:
from google.colab import files

shutil.make_archive("/content/round2_results", "zip", OUT)
files.download("/content/round2_results.zip")